In [ ]:
# Install required Google Cloud packages (commented out as these are typically one-time setup commands)
!pip install gcloud
!gcloud auth application-default login


In [1]:
import pandas as pd                # Data manipulation and analysis
import numpy as np                 # Numerical computing
import time                        # Time-related functions
import os                          # Operating system interfaces
import pandas_gbq                  # Pandas integration with BigQuery
from google.cloud import bigquery  # BigQuery client library
import glob                        # File path pattern matching
import openpyxl                    # Excel file handling
import csv                         # CSV file handling
import re

c:\Users\ana.sales_republica\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Tratamento

In [2]:
diretorio = 'G:\\Drives compartilhados\\República.org\\02. Áreas\\Dados e Conhecimento\\415 - Repositório de Dados\\Repositório Local\\PNAD'

In [3]:
os.chdir(diretorio)  

In [4]:
os.listdir(diretorio)

['2025', '2024', 'pnad_indicadores_serie_2016_2025.xlsx']

Quantidade e porcentagem de pessoas que trabalham no setor público por raça e gênero em posição de liderança 


In [16]:
df = pd.read_excel('pnad_indicadores_serie_2016_2025.xlsx', sheet_name='indicador_07')
df["freq"] = df["freq"].round().astype("Int64")
df["prop"] = (df["prop"] * 100).round(2)
df = df.drop(columns=['freq_se', 'freq_cv','prop_se', 'prop_cv'])
df = df.rename(columns= {'freq':'quantidade_vinculos', 'sexo':'genero','cor':'cor_raca'})   
df = df[['ano','genero','cor_raca','quantidade_vinculos']]              
df.head(6)

,ano,genero,cor_raca,quantidade_vinculos
0,2016,Homem,Branca,150918
1,2016,Homem,Negra,88488
2,2016,Homem,Outra,1627
3,2016,Mulher,Branca,96823
4,2016,Mulher,Negra,54586
5,2016,Mulher,Outra,839


In [23]:
df_genero = (
    df.groupby(["ano", "genero"], as_index=False)
      .agg({"quantidade_vinculos": "sum"})
)

# calcular proporção dentro do ano
df_genero["prop_genero_ano"] = (
    df_genero["quantidade_vinculos"] /
    df_genero.groupby("ano")["quantidade_vinculos"].transform("sum")
) * 100

df_genero.head()

,ano,genero,quantidade_vinculos,prop_genero_ano
0,2016,Homem,241033,61.287731
1,2016,Mulher,152248,38.712269
2,2017,Homem,220551,60.82756
3,2017,Mulher,142033,39.17244
4,2018,Homem,235469,61.435083


In [34]:
df_cor = (
    df.groupby(["ano", "genero", "cor_raca"], as_index=False)
       .agg({"quantidade_vinculos": "sum"})
)

# calcular total por ano
df_cor["total_ano"] = (
    df_cor.groupby("ano")["quantidade_vinculos"].transform("sum")
)

# calcular proporção correta
df_cor["prop_genero_cor_ano"] = (
    df_cor["quantidade_vinculos"] / df_cor["total_ano"]
) * 100

df_cor.head(60)

,ano,genero,cor_raca,quantidade_vinculos,total_ano,prop_genero_cor_ano
0,2016,Feminino,Branca,96823,393281,24.619293
1,2016,Feminino,Negra,54586,393281,13.879643
2,2016,Feminino,Outra,839,393281,0.213333
3,2016,Masculino,Branca,150918,393281,38.374089
4,2016,Masculino,Negra,88488,393281,22.499943
5,2016,Masculino,Outra,1627,393281,0.413699
6,2017,Feminino,Branca,92900,362584,25.621649
7,2017,Feminino,Negra,45760,362584,12.620524
8,2017,Feminino,Outra,3373,362584,0.930267
9,2017,Masculino,Branca,135176,362584,37.281292


In [24]:
def transformar(nome):
    nome =  re.sub("Homem", "Masculino", nome)
    nome = re.sub("Mulher", "Feminino", nome)
    return nome

In [25]:
df['genero'] = df['genero'].apply(transformar)
df['genero'].unique()

array(['Masculino', 'Feminino'], dtype=object)

In [26]:
df

,ano,genero,cor_raca,quantidade_vinculos,prop_genero_cor_ano,total_ano
0,2016,Masculino,Branca,150918,38.374089,393281
1,2016,Masculino,Negra,88488,22.499943,393281
2,2016,Masculino,Outra,1627,0.413699,393281
3,2016,Feminino,Branca,96823,24.619293,393281
4,2016,Feminino,Negra,54586,13.879643,393281
5,2016,Feminino,Outra,839,0.213333,393281
6,2017,Masculino,Branca,135176,37.281292,362584
7,2017,Masculino,Negra,83407,23.003497,362584
8,2017,Masculino,Outra,1968,0.542771,362584
9,2017,Feminino,Branca,92900,25.621649,362584


# Upload

In [15]:
client = bigquery.Client(project='repositoriodedadosgpsp')

In [22]:
schema = [bigquery.SchemaField('ano', 'INTEGER', description= 'Ano de referência da observação'),
          bigquery.SchemaField('genero', 'STRING', description= 'Gênero autodeclarado ou não'),         
          bigquery.SchemaField('quantidade_vinculos', 'INTEGER', description= 'Número total de vinculos observados'),
          bigquery.SchemaField('prop', 'FLOAT', description= 'Proporção de vínculos em relação ao total naquele ano'),
          ]

dataset_ref = client.dataset('perfil_remuneracao')

table_ref = dataset_ref.table('PNAD_vinculos_genero') 
job_config = bigquery.LoadJobConfig(schema=schema)
job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
job.result()

LoadJob<project=repositoriodedadosgpsp, location=US, id=d47fae60-c2a2-494c-9d0b-6249358d8b9b>